# 🥗 NutriWise AI — LLM-Powered Calorie Tracker

**Course-end project | Google Colab edition**  
A reproducible notebook for local Indian food estimates → USDA FoodData Central → OpenRouter → flagged fallback, with Pydantic validation, a JSON meal log, and Gradio UI.

**Run top to bottom.** Put `healthy_meal_plans.csv` beside the notebook (or upload it when asked). An OpenRouter key is optional for local/API use, but is needed for the LLM stage. **Never submit a notebook with an embedded API key.**

**Important:** All nutrition estimates are illustrative, not clinical advice. The final fallback is a **non-saveable placeholder** and must never enter daily totals. Colab runtime data is temporary unless stored on Google Drive.

## 1 · Install dependencies
Colab supports notebook execution and Gradio directly. Version ranges target Pydantic v2 and Gradio v6; restart the runtime if Colab asks after installation.

In [ ]:
%pip install -q "gradio>=6,<7" "pydantic>=2,<3" pandas requests python-dotenv


## 2 · Imports, environment and paths
No key is hard-coded. You can optionally provide a personal USDA key with `USDA_API_KEY`; otherwise the rate-limited demonstration key is used.

In [ ]:
import os, re, json, uuid, getpass, tempfile
from pathlib import Path
from datetime import datetime, date
from fractions import Fraction
from html import escape
from unittest.mock import patch

import pandas as pd
import requests
import gradio as gr
from pydantic import BaseModel, Field, ConfigDict, ValidationError

PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)
MEAL_FILE = DATA_DIR / "meals.json"
USDA_API_KEY = os.getenv("USDA_API_KEY", "DEMO_KEY")
OPENROUTER_MODEL = os.getenv("OPENROUTER_MODEL", "openrouter/free")
print("Working directory:", PROJECT_DIR)
print("Gradio version:", gr.__version__)


## 3 · Load and inspect the Healthy Meal Plans dataset
The supplied Kaggle-style data has **500 records and 14 columns**, with calorie/macronutrient values on a 0–1 scale. **Do not interpret these columns as kcal or grams.** The dataset is used for coverage and test prompts, **not as a calorie reference standard**.

If the CSV is missing in Colab, the cell offers a file-upload dialog; this is the only upload required.

In [ ]:
dataset_path = PROJECT_DIR / "healthy_meal_plans.csv"
if not dataset_path.exists():
    try:
        from google.colab import files
        print("Upload healthy_meal_plans.csv to continue dataset analysis.")
        files.upload()
    except ImportError:
        print("CSV not found. Place healthy_meal_plans.csv next to this notebook.")

if dataset_path.exists():
    df = pd.read_csv(dataset_path)
    print("Shape:", df.shape)
    print("Unique meals:", df["meal_name"].nunique())
    print("Missing cells:", int(df.isna().sum().sum()))
    print("Duplicate rows:", int(df.duplicated().sum()))
    display(df.head(5))
else:
    df = pd.DataFrame()
    print("Dataset analysis skipped; core tracker remains usable.")


## 4 · Local Indian food reference table
These are **starter assumptions** adapted from the original notebook, not independently verified food-composition measurements. Each entry stores a reference portion in units we can check and scale. For example, coconut chutney is **90 kcal per 2 tbsp**, so **1 tbsp = 45 kcal**.

In [ ]:
FOOD_ROWS = [
    ("Idli", 1, "piece", "1 medium", 58, 2, 12, 0.4),
    ("Plain Dosa", 1, "piece", "1 medium", 168, 4, 28, 5),
    ("Masala Dosa", 1, "piece", "1 medium", 350, 7, 48, 14),
    ("Sambar", 1, "cup", "1 cup", 120, 5, 18, 3),
    ("Coconut Chutney", 2, "tbsp", "2 tbsp", 90, 1, 4, 8),
    ("Chapati", 1, "piece", "1 medium", 110, 3, 20, 2),
    ("Cooked Rice", 1, "cup", "1 cup", 205, 4, 45, 0.4),
    ("Dal", 1, "cup", "1 cup", 180, 9, 25, 5),
    ("Vegetable Biryani", 1, "plate", "1 plate", 430, 9, 65, 15),
    ("Chicken Biryani", 1, "plate", "1 plate", 550, 25, 65, 20),
    ("Pongal", 1, "bowl", "1 bowl", 320, 8, 45, 12),
    ("Upma", 1, "bowl", "1 bowl", 250, 6, 40, 7),
    ("Curd Rice", 1, "bowl", "1 bowl", 280, 7, 42, 9),
    ("Paneer Curry", 1, "cup", "1 cup", 320, 14, 12, 24),
    ("Vegetable Khichdi", 1, "bowl", "1 bowl", 270, 9, 42, 7),
    ("Boiled Egg", 1, "piece", "1 large", 78, 6.3, 0.6, 5.3),
    ("Banana", 1, "piece", "1 medium", 105, 1.3, 27, 0.4),
]
FOOD_COLUMNS = ["food", "reference_qty", "reference_unit", "serving", "calories", "protein", "carbs", "fat"]
indian_food_df = pd.DataFrame(FOOD_ROWS, columns=FOOD_COLUMNS)
FOOD_INDEX = {r["food"]: r for r in indian_food_df.to_dict("records")}
indian_food_df.to_csv(PROJECT_DIR / "indian_food_nutrition.csv", index=False)
display(indian_food_df)
print("Local foods:", len(indian_food_df))


## 5 · Strict structured outputs (Pydantic)
All successful sources return the same keys. Pydantic checks types and non-negative numbers; it **does not confirm nutrition accuracy**.

In [ ]:
class FoodItem(BaseModel):
    food: str
    quantity: float = Field(gt=0)
    serving: str

class NutritionEstimate(BaseModel):
    model_config = ConfigDict(extra="forbid")
    name: str = Field(min_length=1)
    calories: float = Field(ge=0)
    protein: float = Field(ge=0)
    carbs: float = Field(ge=0)
    fat: float = Field(ge=0)
    note: str
    source: str
    items: list[FoodItem] | None = None

try:
    NutritionEstimate(name="invalid", calories=-10, protein=1, carbs=1, fat=1,
                      note="test", source="test")
    raise AssertionError("Negative calories should fail validation")
except ValidationError:
    print("Pydantic correctly rejects negative calories")


## 6 · Natural-language portion parsing
Recognizes aliases, plurals, decimal quantities, fractions (`1/2`), and Unicode fractions (`½`). Longest names are matched first to avoid treating *chicken biryani* as vegetable biryani. The **completeness check rejects** unknown extras and incompatible units, instead of silently omitting them.

In [ ]:
ALIASES = {
    "idlis":"Idli", "idli":"Idli", "plain dosas":"Plain Dosa", "plain dosa":"Plain Dosa",
    "dosas":"Plain Dosa", "dosa":"Plain Dosa", "masala dosas":"Masala Dosa", "masala dosa":"Masala Dosa",
    "sambar":"Sambar", "coconut chutney":"Coconut Chutney",
    "chapatis":"Chapati", "chapati":"Chapati", "rotis":"Chapati", "roti":"Chapati",
    "cooked rice":"Cooked Rice", "dal":"Dal", "vegetable biryani":"Vegetable Biryani",
    "veg biryani":"Vegetable Biryani", "chicken biryani":"Chicken Biryani",
    "pongal":"Pongal", "upma":"Upma", "curd rice":"Curd Rice",
    "paneer curry":"Paneer Curry", "vegetable khichdi":"Vegetable Khichdi",
    "khichdi":"Vegetable Khichdi", "boiled eggs":"Boiled Egg", "boiled egg":"Boiled Egg",
    "bananas":"Banana", "banana":"Banana",
}
NUMBER_PATTERN = r"(?:\d+/\d+|\d+(?:\.\d+)?|half|quarter|½|¼|¾)"
UNIT_PATTERN = r"(?:tablespoons?|tbsp|cups?|bowls?|plates?|pieces?|pcs?)"
UNIT_ALIASES = {"cup":"cup", "cups":"cup", "bowl":"bowl", "bowls":"bowl",
                "plate":"plate", "plates":"plate", "piece":"piece", "pieces":"piece",
                "pc":"piece", "pcs":"piece", "tbsp":"tbsp", "tablespoon":"tbsp", "tablespoons":"tbsp"}
WORD_NUMBERS = {"half":0.5, "quarter":0.25, "½":0.5, "¼":0.25, "¾":0.75}

def parse_quantity(value):
    if value is None: return 1.0
    if value in WORD_NUMBERS: return WORD_NUMBERS[value]
    if "/" in value: return float(Fraction(value))
    return float(value)

def parse_meal_details(description):
    text = (description or "").lower().strip()
    found, occupied = [], []
    for alias in sorted(ALIASES, key=len, reverse=True):
        pattern = (rf"(?<!\w)(?:(?P<qty>{NUMBER_PATTERN})\s*"
                   rf"(?P<unit>{UNIT_PATTERN})?\s+)?(?P<food>{re.escape(alias)})(?!\w)")
        for match in re.finditer(pattern, text):
            a, b = match.span()
            if any(a < y and b > x for x,y in occupied): continue
            quantity = parse_quantity(match.group("qty"))
            if quantity <= 0: continue
            unit = match.group("unit")
            found.append({"food":ALIASES[alias], "quantity":quantity,
                          "unit":UNIT_ALIASES.get(unit) if unit else None,
                          "start":a, "end":b})
            occupied.append((a,b))
    return sorted(found, key=lambda x:x["start"])

def parse_meal(description):
    return [{"food":x["food"], "quantity":x["quantity"], "unit":x["unit"]}
            for x in parse_meal_details(description)]

def local_match_is_complete(description):
    parts = parse_meal_details(description)
    if not parts: return False
    remainder = description.lower()
    for p in reversed(parts):
        remainder = remainder[:p["start"]] + " " + remainder[p["end"]:]
        ref = FOOD_INDEX[p["food"]]
        if p["unit"] is not None and p["unit"] != ref["reference_unit"]:
            return False   # e.g. bowl biryani vs reference plate
    remainder = re.sub(r"\b(with|and|a|an|of|the)\b", " ", remainder)
    remainder = re.sub(r"[\s,;+]+", "", remainder)
    return remainder == ""

for example in ["2 idlis with 1/2 cup sambar", "1 tbsp coconut chutney", "2 bananas"]:
    print(example, "→", parse_meal(example), "complete=", local_match_is_complete(example))


## 7 · Accurate *relative* serving calculations
Local estimates are computed only when all described foods and units match known table entries. Units are scaled relative to each reference serving; **2 tbsp coconut chutney is one serving**, not two. Recipe preparation and actual serving weights still introduce uncertainty.

In [ ]:
NUTRIENTS = ("calories", "protein", "carbs", "fat")

def estimate_local_meal(description):
    if not local_match_is_complete(description):
        return None
    parts = parse_meal_details(description)
    totals = {key:0.0 for key in NUTRIENTS}
    items = []
    for p in parts:
        ref = FOOD_INDEX[p["food"]]
        # An omitted unit means a count of the reference serving.
        factor = (p["quantity"] / ref["reference_qty"] if p["unit"]
                  else p["quantity"])
        for key in NUTRIENTS:
            totals[key] += float(ref[key]) * factor
        items.append(FoodItem(food=p["food"], quantity=p["quantity"],
                              serving=(f"{p['unit']}" if p["unit"] else ref["serving"])))
    return NutritionEstimate(
        name=description, **{k:round(v,2) for k,v in totals.items()},
        note="Approximate local reference portions; recipes, cooking oil and serving sizes vary.",
        source="local_food_table", items=items)

for example in ["2 idlis with 1/2 cup sambar", "1 tbsp coconut chutney",
                "2 bananas", "1 bowl chicken biryani"]:
    r = estimate_local_meal(example)
    print(example, "→", r.calories if r else "not a complete compatible local match")


## 8 · USDA FoodData Central (free API stage)
USDA is used for **simple identifiable foods**, especially explicit gram weights. Search ranking is not reliable enough to automatically choose the first candidate. A conservative description matcher is used; uncertain matches pass to OpenRouter. The demonstration key can hit rate limits.

In [ ]:
USDA_URL = "https://api.nal.usda.gov/fdc/v1/foods/search"

def search_usda(food_name):
    try:
        r = requests.get(USDA_URL,
            params={"api_key":USDA_API_KEY, "query":food_name, "dataType":["Foundation", "SR Legacy"], "pageSize":15},
            timeout=15)
        r.raise_for_status()
        return r.json().get("foods", [])
    except (requests.RequestException, ValueError, TypeError):
        return []

def select_best_usda_match(foods, query):
    q = query.strip().lower()
    # Only choose a clearly corresponding name, not fuzzy unrelated dishes.
    forms = {q, q+"s", q[:-1] if q.endswith("s") else q}
    forbidden = ("dehydrated", "powder", "dried", "canned", "juice", "flour")
    for food in foods:
        desc = food.get("description", "").strip().lower()
        if desc in forms:
            return food
    for food in foods:
        desc = food.get("description", "").strip().lower()
        if any(bad in desc for bad in forbidden): continue
        if desc.endswith(", raw") and desc[:-5].strip() in forms:
            return food
    return None

def extract_usda_nutrition(food, grams=100.0):
    nutrients = {}
    for n in food.get("foodNutrients", []):
        nutrient_id, value = n.get("nutrientId"), n.get("value")
        if nutrient_id is not None and value is not None:
            nutrients[int(nutrient_id)] = float(value)
    # 1008 (energy), 2047/2048 (alternate energy); macro IDs are USDA standard.
    energy = next((nutrients[k] for k in (1008,2047,2048) if k in nutrients), None)
    if energy is None or not all(k in nutrients for k in (1003,1005,1004)):
        return None
    f = grams / 100.0
    return NutritionEstimate(
        name=food.get("description", "USDA food"), calories=round(energy*f,2),
        protein=round(nutrients[1003]*f,2), carbs=round(nutrients[1005]*f,2),
        fat=round(nutrients[1004]*f,2),
        note=f"USDA FDC ID {food.get('fdcId','unknown')}; assumed {grams:g} g. Verify food and serving match.",
        source="usda_fooddata_central")

SIMPLE_GRAM_PATTERN = re.compile(r"(?:(?P<grams>\d+(?:\.\d+)?)\s*(?:g|grams?)\s+)?(?P<food>[a-zA-Z][a-zA-Z -]*)$", re.I)

def estimate_usda(description):
    m = SIMPLE_GRAM_PATTERN.fullmatch(description.strip())
    if not m or re.search(r"\b(with|and|homemade|curry|salad|soup|stew)\b", description, re.I):
        return None
    grams = float(m.group("grams")) if m.group("grams") else 100.0
    if grams <= 0 or grams > 2000: return None
    name = m.group("food").strip()
    selected = select_best_usda_match(search_usda(name), name)
    return extract_usda_nutrition(selected, grams) if selected else None

print("USDA lookup ready (network requests occur only when called).")


## 9 · OpenRouter LLM fallback
Enter the key **privately** with `getpass` in a running Colab session. The `openrouter/free` router can change models between calls; results will vary. If there is no key, requests fail, or returned JSON is invalid, the pipeline moves to the non-saveable placeholder.

**No API key is stored in the notebook output.**

In [ ]:
# Optional: run this cell to enable the OpenRouter fallback.
# Leave it blank to test local/USDA behavior without an LLM.
if not os.getenv("OPENROUTER_API_KEY"):
    key = getpass.getpass("OpenRouter API key (press Enter to skip): ").strip()
    if key:
        os.environ["OPENROUTER_API_KEY"] = key
    del key
print("OpenRouter configured:", bool(os.getenv("OPENROUTER_API_KEY")))


In [ ]:
OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"

def estimate_with_openrouter(description):
    key = os.getenv("OPENROUTER_API_KEY")
    if not key: return None
    prompt = ("Estimate the nutrition of the described meal using a plausible portion and recipe. "
              "Return ONLY a JSON object with fields name, calories, protein, carbs, fat, note, source. "
              "Calories in kcal; macros in grams. Explain serving assumptions and uncertainty in note. "
              "Do not invent a precise measurement. Set source to openrouter. Meal: " + description)
    try:
        r = requests.post(OPENROUTER_URL,
            headers={"Authorization":f"Bearer {key}", "Content-Type":"application/json"},
            json={"model":OPENROUTER_MODEL,
                  "messages":[{"role":"system", "content":"You are a nutrition estimation assistant. Return valid JSON only."},
                              {"role":"user", "content":prompt}],
                  "temperature":0.2, "max_tokens":2000}, timeout=90)
        r.raise_for_status()
        payload = r.json()
        choice = (payload.get("choices") or [])[0]
        content = (choice.get("message") or {}).get("content")
        if not isinstance(content,str) or not content.strip(): return None
        content = re.sub(r"^```(?:json)?\s*|\s*```$", "", content.strip(), flags=re.I)
        parsed = json.loads(content)
        if not isinstance(parsed,dict): return None
        parsed["source"] = "openrouter"
        parsed.setdefault("items", None)
        return NutritionEstimate.model_validate(parsed)
    except (requests.RequestException, ValueError, KeyError, IndexError, TypeError, ValidationError):
        return None

print("OpenRouter fallback ready.")


## 10 · Unified no-stop pipeline
**Local → USDA → OpenRouter → warning placeholder**. An empty input produces a validation message rather than fake nutrition. The final fallback uses demonstration numbers only, is **not considered a reliable estimate**, and **cannot be saved**.

In [ ]:
def rough_fallback(description):
    return NutritionEstimate(
        name=description, calories=250.0, protein=10.0, carbs=30.0, fat=10.0,
        note=("LOW CONFIDENCE: arbitrary demonstration placeholder only; not measured, "
              "not ingredient-based, and not suitable for dietary tracking."),
        source="rough_fallback")

def estimate_nutrition(description):
    text = (description or "").strip()
    if not text: return None
    for stage in (estimate_local_meal, estimate_usda, estimate_with_openrouter):
        try:
            value = stage(text)
            if value is not None:
                return NutritionEstimate.model_validate(value)
        except Exception as exc:
            print("Estimation stage unavailable:", stage.__name__, type(exc).__name__)
    return rough_fallback(text)

print("Unified pipeline ready.")


## 11 · Local JSON persistence and daily totals
UUIDs stay internal. Records have an ISO local date and timestamp and are prepended to `data/meals.json`. Writing through a temporary file prevents a partially written JSON file. Colab's runtime folder is ephemeral; copy `data/meals.json` to Drive for persistence beyond the current session.

In [ ]:
def load_meals():
    if not MEAL_FILE.exists(): return []
    try:
        records = json.loads(MEAL_FILE.read_text(encoding="utf-8"))
        return records if isinstance(records,list) else []
    except (OSError, json.JSONDecodeError):
        print("Meal file unreadable; showing an empty list. Restore the backup if needed.")
        return []

def _write_meals(records):
    MEAL_FILE.parent.mkdir(parents=True, exist_ok=True)
    tmp = MEAL_FILE.with_suffix(".tmp")
    tmp.write_text(json.dumps(records, indent=2, ensure_ascii=False), encoding="utf-8")
    os.replace(tmp, MEAL_FILE)

def save_meal(estimate):
    value = NutritionEstimate.model_validate(estimate)
    if value.source == "rough_fallback":
        raise ValueError("Placeholders cannot be saved")
    now = datetime.now().astimezone()
    record = {"id":str(uuid.uuid4()), "date":now.date().isoformat(),
              "created_at":now.isoformat(), **value.model_dump()}
    _write_meals([record]+load_meals())
    return record

def delete_meal(meal_id):
    records = load_meals()
    keep = [x for x in records if x.get("id") != meal_id]
    if len(keep) == len(records): return False
    _write_meals(keep)
    return True

def get_todays_meals():
    today = datetime.now().astimezone().date().isoformat()
    return [m for m in load_meals() if m.get("date") == today]

def get_daily_total():
    return round(sum(float(m.get("calories",0)) for m in get_todays_meals()),2)

print("Meal storage ready:", MEAL_FILE)


## 12 · Automated offline tests
These checks need **no API key and no external requests**. They cover reference calculations, fractions, tablespoons, unknown ingredients, serving mismatch, API/LLM outages, negative-value validation and JSON add/delete persistence. The final fallback is explicitly rejected by storage.

In [ ]:
def run_offline_tests():
    expectations = {
        "2 idlis with 1/2 cup sambar":176.0,
        "1.5 cups cooked rice":307.5,
        "3 boiled eggs":234.0,
        "1 masala dosa with coconut chutney":440.0,
        "2 bananas":210.0,
        "1 cup dal with 2 chapatis":400.0,
        "1 tbsp coconut chutney":45.0,
        "2 tbsp coconut chutney":90.0,
    }
    for meal, expected in expectations.items():
        result = estimate_local_meal(meal)
        assert result is not None and abs(result.calories-expected)<0.001, (meal,result)
    assert estimate_local_meal("2 chapatis with paneer curry and salad") is None
    assert estimate_local_meal("1 bowl chicken biryani") is None
    assert estimate_local_meal("pizza with cheese") is None
    try:
        NutritionEstimate(name="bad", calories=-2, protein=2, carbs=2, fat=2, note="", source="test")
        raise AssertionError("Pydantic accepted negative calories")
    except ValidationError:
        pass
    with patch.dict(estimate_nutrition.__globals__,
                    {"search_usda": lambda name: [],
                     "estimate_with_openrouter": lambda description: None}):
        result = estimate_nutrition("100 g spinach")
        assert result.source == "rough_fallback"
    global MEAL_FILE
    original_path = MEAL_FILE
    try:
        with tempfile.TemporaryDirectory() as td:
            MEAL_FILE = Path(td)/"meals.json"
            item = save_meal(estimate_local_meal("2 idlis with 1 cup sambar"))
            assert len(get_todays_meals())==1 and get_daily_total()==236
            assert delete_meal(item["id"])
            assert len(get_todays_meals())==0 and get_daily_total()==0
            try:
                save_meal(rough_fallback("unknown"))
                raise AssertionError("Placeholder was saved")
            except ValueError:
                pass
    finally:
        MEAL_FILE = original_path
    print(f"PASS: {len(expectations)} local estimates + 3 mismatch checks + validation + outage + storage tests")

run_offline_tests()


## 13 · Kaggle coverage analysis (no LLM calls)
Coverage is intentionally low because the local table focuses on Indian meals, whereas the Kaggle meal names cover a different cuisine mix. This is a **coverage check**, not accuracy evaluation; the Kaggle normalized calorie numbers cannot serve as kcal ground truth.

In [ ]:
if not df.empty:
    distinct_meals = sorted(df["meal_name"].dropna().astype(str).unique())
    coverage_rows = [{"meal_name":m, "local_match":estimate_local_meal(m) is not None}
                     for m in distinct_meals]
    coverage_df = pd.DataFrame(coverage_rows)
    print(f"Distinct meal names: {len(distinct_meals)}")
    print(f"Local full-match coverage: {coverage_df['local_match'].mean():.1%}")
    print(f"Healthy label share: {df['is_healthy'].mean():.1%}")
    display(coverage_df)
    print("Kaggle nutrition features are normalized; no quantitative kcal accuracy score is computed.")
else:
    print("Upload CSV to run dataset coverage analysis.")


## 14 · User-friendly Gradio calorie tracker
The visible interface hides UUIDs, shows a nutrition summary, lets users save an estimate, and delete a selected meal via a friendly dropdown. The evaluator can inspect the validated JSON in **Technical Details**. Placeholder estimates are not saveable.

In [ ]:
def nutrition_card(data):
    if not data: return "### Enter a meal to see its nutrition."
    if data.get("source")=="rough_fallback":
        return ("### ⚠️ Estimate unavailable\n\nAll estimators failed. "
                "The technical placeholder is **not a nutrition result** and cannot be saved.")
    return (f"### 🍽️ {escape(data['name'])}\n\n"
            f"| Calories | Protein | Carbs | Fat |\n|---|---|---|---|\n"
            f"| **{data['calories']:.0f} kcal** | **{data['protein']:.1f} g** | "
            f"**{data['carbs']:.1f} g** | **{data['fat']:.1f} g** |\n\n"
            f"*{escape(data['note'])}*\n\n**Source:** {escape(data['source'])}")

def dashboard_ui():
    meals = get_todays_meals()
    sums = {k:sum(float(m.get(k,0)) for m in meals) for k in NUTRIENTS}
    return (f"### 📊 Today's Nutrition\n\n# {sums['calories']:.0f} kcal\n\n"
            f"**{len(meals)} meals logged**\n\n"
            f"Protein: **{sums['protein']:.1f} g** · Carbs: **{sums['carbs']:.1f} g** · Fat: **{sums['fat']:.1f} g**")

def meal_list_ui():
    meals = get_todays_meals()
    if not meals: return "No meals saved today."
    return "\n\n---\n\n".join(
        f"**{escape(m['name'])}** — {float(m['calories']):.0f} kcal  \n"
        f"{escape(m['created_at'][11:16])} · {escape(m['source'])}"
        for m in meals)

def dropdown_ui():
    return gr.update(choices=[
        (f"{m['name']} ({float(m['calories']):.0f} kcal) — {m['created_at'][11:16]}", m['id'])
        for m in get_todays_meals()], value=None)

def estimate_meal_ui(description):
    if not description or not description.strip():
        return "Please describe your meal.", None, None, "Enter a meal description."
    try:
        result = estimate_nutrition(description)
        data = result.model_dump() if result else None
        saveable = data if data and data["source"]!="rough_fallback" else None
        msg = "Nutrition estimated. You can save this meal." if saveable else "No reliable estimate; nothing to save."
        return nutrition_card(data), saveable, data, msg
    except Exception:
        return nutrition_card(None), None, None, "Estimation unavailable. Please try again."

def save_meal_ui(data):
    if not data: status = "Estimate a meal first (placeholders cannot be saved)."
    else:
        try:
            save_meal(NutritionEstimate.model_validate(data))
            status = "✅ Meal saved."
        except Exception:
            status = "Unable to save this meal."
    return status, dashboard_ui(), meal_list_ui(), dropdown_ui(), None

def delete_meal_ui(meal_id):
    status = ("✅ Meal deleted." if delete_meal(meal_id) else "Please select a saved meal.") if meal_id else "Please select a meal."
    return status, dashboard_ui(), meal_list_ui(), dropdown_ui()

print("Gradio functions ready.")


In [ ]:
def build_app():
    with gr.Blocks(title="NutriWise AI") as demo:
        gr.Markdown("# 🥗 NutriWise AI\n### Your Smart Nutrition Companion")
        dashboard = gr.Markdown(value=dashboard_ui())
        meal_input = gr.Textbox(label="What did you eat?", placeholder="2 idlis with 1 cup sambar", lines=2)
        with gr.Row():
            estimate_btn = gr.Button("🔍 Estimate Nutrition", variant="primary")
            save_btn = gr.Button("💾 Save Meal")
        nutrition_output = gr.Markdown(value=nutrition_card(None))
        status = gr.Markdown()
        current_estimate = gr.State(None)
        with gr.Accordion("Technical Details (validated JSON)", open=False):
            technical_output = gr.JSON()
        gr.Markdown("## 🍱 Today's Meals")
        history = gr.Markdown(value=meal_list_ui())
        delete_dropdown = gr.Dropdown(label="Select a saved meal to remove", choices=[], interactive=True)
        with gr.Row():
            delete_btn = gr.Button("🗑️ Delete Selected Meal", variant="stop")
            refresh_btn = gr.Button("🔄 Refresh")
        estimate_btn.click(estimate_meal_ui, inputs=meal_input,
            outputs=[nutrition_output, current_estimate, technical_output, status])
        save_btn.click(save_meal_ui, inputs=current_estimate,
            outputs=[status, dashboard, history, delete_dropdown, current_estimate])
        delete_btn.click(delete_meal_ui, inputs=delete_dropdown,
            outputs=[status, dashboard, history, delete_dropdown])
        refresh_btn.click(lambda:(dashboard_ui(),meal_list_ui(),dropdown_ui()),
            outputs=[dashboard,history,delete_dropdown])
        demo.load(lambda:(dashboard_ui(),meal_list_ui(),dropdown_ui()),
            outputs=[dashboard,history,delete_dropdown])
    return demo

print("NutriWise app builder ready.")


## 15 · Launch the application in Colab
Only the last cell starts a server. **Don't relaunch multiple instances simultaneously.** If you need to stop it, run `demo.close()`. Shared `.gradio.live` links are temporary and should not be treated as production deployment. In Gradio 6, the theme belongs to `launch()`.

Use the **generated link from this run**, not an old Gradio link.

In [ ]:
demo = build_app()
demo.launch(share=True, inline=False, debug=False, theme=gr.themes.Soft(), show_error=True)


## 16 · Submission checklist and limitations

- [ ] Run all cells in a **fresh Colab runtime** (the final launch cell last).
- [ ] Enter an OpenRouter API key privately if you need to demonstrate the LLM fallback.
- [ ] Test three example sources: `2 idlis with 1 cup sambar` (local), `100 g spinach` (USDA, if reachable), and `homemade mushroom quinoa curry` (OpenRouter, if available).
- [ ] Save, refresh, and delete a meal; check daily totals; confirm rough placeholders cannot be saved.
- [ ] Capture **real screenshots from your Colab Gradio run** for the report.
- [ ] Include this notebook, CSV, code export, setup instructions, and HTML documentation in Google Drive.

**Limitations:** approximate starter table, USDA food-match/portion uncertainty, free-model variability, normalized Kaggle features without invertible scaling, API limits, and Colab runtime storage. The notebook is a demonstration, not medical or dietary advice. The course PDF also calls for modular source files and a final report/screenshots; the companion package supplies code exports, but actual screenshots must be captured from your run.